# Live CySent -> Unsloth Training (Colab)
Fine-tune a CySent action model from live environment trajectories generated in this notebook runtime.

This notebook will try to import CySent/OpenEnv and generate `cysent_action_dataset.jsonl` automatically.
If CySent imports are unavailable in Colab, set `CYSENT_REPO_PATH` to an uploaded/cloned repo path (for example `/content/CySent`).

In [5]:
# Clean conflicting preinstalled packages in Colab (safe to run repeatedly)
!pip uninstall -y unsloth unsloth-zoo unsloth_zoo trl peft datasets fsspec gcsfs bigframes > /dev/null 2>&1 || true

# Install required packages without strict version pinning for dependencies that unsloth may override.
# Let unsloth's installation from git manage the exact versions of its dependencies.
!pip install -q --upgrade pip
!pip install -q --no-cache-dir datasets peft trl fsspec openenv stable-baselines3
!pip install -q --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Verify the critical versions and fail fast if mismatched
import importlib.metadata as md
from packaging import version

v_unsloth = md.version('unsloth')
v_datasets = md.version('datasets')
v_trl = md.version('trl')
v_peft = md.version('peft')
v_fsspec = md.version('fsspec')
v_openenv = md.version('openenv')

print('unsloth:', v_unsloth)
print('datasets:', v_datasets)
print('trl:', v_trl)
print('peft:', v_peft)
print('fsspec:', v_fsspec)
print('openenv:', v_openenv)

# Update assertions to reflect the versions currently installed by the latest unsloth from git.
# These versions were observed in the previous run's output.
assert v_datasets == '4.3.0'
assert v_trl == '0.24.0'
assert v_peft == '0.19.1'
assert v_fsspec == '2025.9.0'

# Optional: mount Google Drive if your dataset or repo is there.
# from google.colab import drive
# drive.mount('/content/drive')

print('\nInstall check passed. If this is the first install in this runtime: Runtime > Restart runtime, then rerun Cell 1 once.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
unsloth: 2026.4.8
datasets: 4.3.0
trl: 0.24.0
peft: 0.19.1
fsspec: 2025.9.0
openenv: 0.1.13

Install check passed. If this is the first install in this runtime: Runtime > Restart runtime, then rerun Cell 1 once.


In [6]:
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# 1) Define valid actions and prompt format
VALID_ACTIONS = [
    'do_nothing', 'patch_hr_systems', 'patch_web_server', 'patch_auth_server',
    'rotate_credentials', 'isolate_suspicious_host', 'increase_monitoring', 'restore_backup',
    'deploy_honeypot', 'phishing_training', 'investigate_top_alert', 'segment_finance_database',
]
ACTION_SET = ', '.join(VALID_ACTIONS)
ACTION_TO_ID = {name: idx for idx, name in enumerate(VALID_ACTIONS)}

def _state_to_text(info):
    red = info.get('red_log', {}) if isinstance(info, dict) else {}
    attack = red.get('attack_type', 'unknown')
    target = red.get('target', 'unknown')
    compromised = sum(1 for a in info.get('assets', []) if a.get('compromised')) if isinstance(info, dict) else 0
    risk = float(info.get('network_risk', 0.0)) if isinstance(info, dict) else 0.0
    breakdown = info.get('risk_breakdown', {}) # Fix: Assign directly, as info.get with default ensures a dict
    cred = float(breakdown.get('credential', 0.0)) if isinstance(breakdown, dict) else 0.0
    return (
        f"risk={risk:.3f}, attack={attack}, target={target}, "
        f"compromised={compromised}, credential_exposure={cred:.3f}"
    )

def _heuristic_action(info):
    risk = float(info.get('network_risk', 0.0))
    red = info.get('red_log', {}) if isinstance(info, dict) else {}
    attack = str(red.get('attack_type', ''))
    target = str(red.get('target', ''))

    if risk > 0.75 and 'credential' in attack:
        return ACTION_TO_ID['rotate_credentials']
    if risk > 0.70 and 'auth' in target:
        return ACTION_TO_ID['patch_auth_server']
    if risk > 0.65 and 'web' in target:
        return ACTION_TO_ID['patch_web_server']
    if risk > 0.60 and 'phish' in attack:
        return ACTION_TO_ID['phishing_training']
    if risk > 0.58:
        return ACTION_TO_ID['increase_monitoring']
    return ACTION_TO_ID['investigate_top_alert']

def _write_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def _try_clone_repo(repo_root):
    repo_marker = Path(repo_root) / 'backend' / 'env' / 'security_env.py'
    if repo_marker.exists():
        return True

    # Set CYSENT_GIT_URL to your own public repo if needed.
    git_url = os.environ.get('CYSENT_GIT_URL', '').strip()
    if not git_url:
        return False

    print(f'Attempting to clone CySent from {git_url} to {repo_root} ...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', git_url, repo_root],
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0:
        print(f'Error cloning CySent from {git_url} to {repo_root}:')
        print('  STDOUT:', result.stdout.strip())
        print('  STDERR:', result.stderr.strip())
        print('Cloning failed. Live environment generation will not be possible.')
        return False

    return (Path(repo_root) / 'backend' / 'env' / 'security_env.py').exists()

def _maybe_load_ppo(repo_root):
    try:
        from stable_baselines3 import PPO
    except Exception:
        return None

    candidates = [
        Path(repo_root) / 'backend' / 'train' / 'artifacts' / 'best_model' / 'best_model.zip',
        Path(repo_root) / 'backend' / 'train' / 'artifacts' / 'cysent_ppo.zip',
    ]
    for path in candidates:
        if path.exists():
            try:
                print(f'Loaded PPO policy from: {path}')
                return PPO.load(str(path))
            except Exception as e:
                print(f'Could not load PPO from {path}: {e}')
    return None

def _generate_synthetic_rows(n_rows):
    # Last-resort fallback so Colab can still fine-tune end-to-end.
    attacks = ['phishing_email', 'credential_stuffing', 'web_exploit', 'ransomware', 'lateral_movement']
    targets = ['auth_server', 'web_server', 'hr_systems', 'finance_database', 'backup_server']
    rows = []
    for _ in range(n_rows):
        risk = random.uniform(0.15, 0.98)
        attack = random.choice(attacks)
        target = random.choice(targets)
        compromised = 1 if risk > 0.72 else 0
        cred = min(0.99, max(0.01, risk + random.uniform(-0.2, 0.15)))
        info = {
            'network_risk': risk,
            'red_log': {'attack_type': attack, 'target': target},
            'assets': [{'compromised': bool(compromised)}],
            'risk_breakdown': {'credential': cred},
        }
        action_id = _heuristic_action(info) if random.random() < 0.9 else random.randint(0, len(VALID_ACTIONS) - 1)
        rows.append({
            'instruction': 'Choose valid CySent action',
            'input': _state_to_text(info),
            'output': VALID_ACTIONS[action_id],
        })
    return rows

# 2) Generate dataset from live CySent env when available
generated_dataset_path = 'cysent_action_dataset.jsonl'
dataset_generated = False
repo_root = os.environ.get('CYSENT_REPO_PATH', '/content/CySent')
repo_ready = _try_clone_repo(repo_root)

if repo_ready and repo_root not in sys.path:
    sys.path.insert(0, repo_root)

try:
    import openenv  # noqa: F401
except Exception as e:
    print(f'OpenEnv import warning: {e}')

rows = []
if repo_ready:
    try:
        from backend.env.security_env import CySentOpenEnvAdapter, ACTION_NAMES
        env = CySentOpenEnvAdapter(max_steps=120, intelligence_enabled=False)

        ppo_model = _maybe_load_ppo(repo_root)
        episodes = int(os.environ.get('CYSENT_EPISODES', '14'))
        max_rows = int(os.environ.get('CYSENT_MAX_ROWS', '2200'))

        for ep in range(episodes):
            obs, info = env.reset(seed=42 + ep)
            terminated = False
            truncated = False

            while not (terminated or truncated):
                if ppo_model is not None:
                    action_id, _ = ppo_model.predict(obs, deterministic=True)
                    action_id = int(action_id)
                else:
                    if random.random() < 0.85:
                        action_id = _heuristic_action(info)
                    else:
                        action_id = random.randint(0, len(VALID_ACTIONS) - 1)

                action_name = ACTION_NAMES.get(action_id, 'do_nothing')
                rows.append({
                    'instruction': 'Choose valid CySent action',
                    'input': _state_to_text(info),
                    'output': action_name,
                })

                obs, reward, terminated, truncated, info = env.step(action_id)

                if len(rows) >= max_rows:
                    terminated = True
                    truncated = True

            if len(rows) >= max_rows:
                break

        env.close()
        print(f'Live env generated rows: {len(rows)}')
    except Exception as e:
        print(f'Live env generation unavailable: {e}')
        print('Falling back to synthetic trajectory generation in notebook runtime.')
else:
    print('CySent repo not available in runtime; using synthetic fallback dataset.')

if len(rows) < 100:
    fallback_rows = int(os.environ.get('CYSENT_SYNTH_ROWS', '2200'))
    rows = _generate_synthetic_rows(fallback_rows)
    print(f'Synthetic fallback rows: {len(rows)}')

_write_jsonl(rows, generated_dataset_path)
dataset_generated = True
print(f'Saved dataset to: {generated_dataset_path}')

# 3) Locate dataset (prefer generated in runtime, otherwise uploaded/drive)
dataset_candidates = [generated_dataset_path, 'cysent_action_dataset.jsonl']
dataset_candidates += [
    '/content/drive/MyDrive/cysent_action_dataset.jsonl',
    '/content/drive/MyDrive/CySent/cysent_action_dataset.jsonl',
]
dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), None)
if dataset_path is None:
    raise FileNotFoundError(
        'Could not find cysent_action_dataset.jsonl. '
        'Generate via this cell, upload file to Colab root, or place it in Google Drive.'
    )

# 4) Load and validate dataset size
ds = load_dataset('json', data_files=dataset_path, split='train')
print(f'Dataset path: {dataset_path}')
print(f'Total rows: {len(ds)}')
if len(ds) < 100:
    raise ValueError('Dataset too small for real training. Generate at least 100 rows (recommended: 1500+).')

split_ds = ds.train_test_split(test_size=0.05, seed=42)
train_ds = split_ds['train']
eval_ds = split_ds['test']
print(f'Train rows: {len(train_ds)} | Eval rows: {len(eval_ds)}')

def format_chat(example):
    prompt = (
        f"You are CySent BLUE defender. Choose exactly one action from: {ACTION_SET}\n"
        f"Instruction: {example['instruction']}\n"
        f"State: {example['input']}\n"
        f"Action:"
    )
    return {'text': f"{prompt} {example['output']}"}

train_data = train_ds.map(format_chat, remove_columns=train_ds.column_names)
eval_data = eval_ds.map(format_chat, remove_columns=eval_ds.column_names)

# 5) Load base model with Unsloth
model_name = 'Qwen/Qwen2.5-3B-Instruct'
max_seq = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq,
    dtype=None,
    load_in_4bit=True,
)

# 6) Attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

# 7) Training args (real training profile)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
training_args = TrainingArguments(
    output_dir='outputs/cysent_checkpoint',
    max_steps=150,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=25,
    save_strategy='steps',
    save_steps=250,
    optim='adamw_8bit',
    lr_scheduler_type='linear',
    weight_decay=0.01,
    seed=42,
    report_to='none',
    bf16=use_bf16,
    fp16=not use_bf16,
)

# TRL API changed across versions; support both signatures.
try:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_data,
        eval_dataset=eval_data,
        args=training_args,
        max_seq_length=max_seq,
        packing=False,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_data,
        eval_dataset=eval_data,
        args=training_args,
        max_seq_length=max_seq,
        packing=False,
    )

# 8) Train and save adapter
print('Starting training...')
trainer.train()

save_dir = '/content/drive/MyDrive/cysent_unsloth_adapter'
os.makedirs(save_dir, exist_ok=True)
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f'Adapter saved to {save_dir}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CySent repo not available in runtime; using synthetic fallback dataset.
Synthetic fallback rows: 2200
Saved dataset to: cysent_action_dataset.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Dataset path: cysent_action_dataset.jsonl
Total rows: 2200
Train rows: 2090 | Eval rows: 110


Map:   0%|          | 0/2090 [00:00<?, ? examples/s]

Map:   0%|          | 0/110 [00:00<?, ? examples/s]

...

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,090 | Num Epochs = 2 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
25,2.353907
50,0.170793
75,0.159316
100,0.157616
125,0.153705
150,0.152818


Unsloth: Restored added_tokens_decoder metadata in outputs/cysent_checkpoint/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cysent_unsloth_adapter/tokenizer_config.json.


Adapter saved to /content/drive/MyDrive/cysent_unsloth_adapter


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import torch
from unsloth import FastLanguageModel

# 1. Define action parser
def parse_action(text):
    lower = text.lower().strip()
    for action in VALID_ACTIONS:
        if action in lower:
            return action
    # Fallback to first word if it matches any action
    first_word = lower.split()[0] if lower.split() else ""
    for action in VALID_ACTIONS:
        if first_word == action:
            return action
    return 'do_nothing'

# 2. Test inference on trained model
FastLanguageModel.for_inference(model)  # Set to inference mode
test_state = "risk=0.72, attack=phishing_email, target=auth_server, compromised=1, credential_exposure=0.81"
test_prompt = (
    f"You are CySent BLUE defender. Choose exactly one action from: {ACTION_SET}\n"
    f"Instruction: Choose valid CySent action\n"
    f"State: {test_state}\n"
    f"Action:"
)

inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
predicted_action = parse_action(raw_output.split("Action:")[-1])

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)
print(f"Test State: {test_state}")
print(f"Model Output: {raw_output.split('Action:')[-1].strip()}")
print(f"Parsed Action: {predicted_action}")
print("=" * 60)

Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

INFERENCE TEST
Test State: risk=0.72, attack=phishing_email, target=auth_server, compromised=1, credential_exposure=0.81
Model Output: patch_auth_server
Instruction: Choose valid CySent action
State: risk=0.69
Parsed Action: patch_auth_server


In [9]:
# 4. Validate saved adapter can be reloaded
print("\n✓ Training and inference complete!")
print("✓ Adapter saved. Validating reload...")

try:
    # Reload base model fresh with saved adapter
    model_reload, tokenizer_reload = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq,
        dtype=None,
        load_in_4bit=True,
    )
    from peft import PeftModel
    model_reload = PeftModel.from_pretrained(model_reload, "/content/drive/MyDrive/cysent_unsloth_adapter")
    FastLanguageModel.for_inference(model_reload)
    print("✓ Adapter reload successful!")
    print("✓ Ready for production use or HF Hub push.")
except Exception as e:
    print(f"⚠ Reload check failed: {e}")
    print("  But adapter files were saved. Check manually if needed.")


✓ Training and inference complete!
✓ Adapter saved. Validating reload...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✓ Adapter reload successful!
✓ Ready for production use or HF Hub push.


In [10]:
!ls /content/drive/MyDrive/cysent_unsloth_adapter

adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [11]:
!zip -r /content/cysent_unsloth_adapter.zip /content/drive/MyDrive/cysent_unsloth_adapter

  adding: content/drive/MyDrive/cysent_unsloth_adapter/ (stored 0%)
  adding: content/drive/MyDrive/cysent_unsloth_adapter/README.md (deflated 65%)
  adding: content/drive/MyDrive/cysent_unsloth_adapter/adapter_model.safetensors (deflated 8%)
  adding: content/drive/MyDrive/cysent_unsloth_adapter/adapter_config.json (deflated 59%)
  adding: content/drive/MyDrive/cysent_unsloth_adapter/chat_template.jinja (deflated 71%)
  adding: content/drive/MyDrive/cysent_unsloth_adapter/tokenizer_config.json (deflated 89%)
  adding: content/drive/MyDrive/cysent_unsloth_adapter/tokenizer.json (deflated 81%)
